# P01：数据清洗与存储| 项目 | 内容 ||------|------|| 课程 | 数据分析与经济决策（ds2026） || 姓名 | 柯骋 || 学号 | 25210150 || 日期 | 2026-05-21 |

## 任务说明本Notebook完成以下数据清洗与管理工作：1. 单表清洗：缺失值、日期格式、数据类型、重复值、离群值2. 宽表与长表转换3. 多表合并4. CSV存储（方式A）5. SQLite存储（方式C，进阶）

## 1. 加载原始数据

In [ ]:
import osimport pandas as pdimport numpy as npimport sqlite3BASE_DIR = os.path.dirname(os.path.abspath("__file__"))STOCKS = [    {"code": "600036", "name": "招商银行", "industry": "银行"},    {"code": "601398", "name": "工商银行", "industry": "银行"},    {"code": "002594", "name": "比亚迪",   "industry": "汽车"},    {"code": "601633", "name": "长城汽车", "industry": "汽车"},    {"code": "000002", "name": "万科A",    "industry": "房地产"},    {"code": "600048", "name": "保利发展", "industry": "房地产"},    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},    {"code": "000858", "name": "五粮液",   "industry": "白酒"},    {"code": "601857", "name": "中国石油", "industry": "能源"},    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},]# 加载所有股票数据stock_data = {}for stock in STOCKS:    filepath = os.path.join(BASE_DIR, "data", "stock", f"stock_{stock['code']}.csv")    if os.path.exists(filepath):        df = pd.read_csv(filepath)        stock_data[stock['code']] = df        print(f"成功加载 {len(stock_data)} 只股票数据")for code, df in stock_data.items():    print(f"  {code}: {df.shape}, 日期范围 {df['日期'].iloc[0]} ~ {df['日期'].iloc[-1]}")

In [ ]:
# 加载指数数据hs300 = pd.read_csv(os.path.join(BASE_DIR, "data", "index", "index_000300.csv"))zz500 = pd.read_csv(os.path.join(BASE_DIR, "data", "index", "index_000905.csv"))# 加载宏观数据cpi = pd.read_csv(os.path.join(BASE_DIR, "data", "macro", "macro_cpi.csv"))m2 = pd.read_csv(os.path.join(BASE_DIR, "data", "macro", "macro_m2.csv"))# 加载财务数据finance = pd.read_csv(os.path.join(BASE_DIR, "data", "finance", "finance_ratios.csv"))print(f"沪深300: {hs300.shape}, 中证500: {zz500.shape}")print(f"CPI: {cpi.shape}, M2: {m2.shape}, 财务: {finance.shape}")

## 2. 单表清洗### 2.1 缺失值检测

In [ ]:
# 检测每只股票的缺失值missing_report = []for code, df in stock_data.items():    for col in df.columns:        n_missing = df[col].isna().sum()        pct = n_missing / len(df) * 100        if n_missing > 0:            missing_report.append({                "code": code, "column": col,                "missing_count": n_missing, "missing_pct": f"{pct:.2f}%"            })if missing_report:    missing_df = pd.DataFrame(missing_report)    print("缺失值报告：")    print(missing_df)else:    print("所有股票数据均无缺失值！")# 可能原因分析：# - 停牌：交易日内股票停牌，成交量和成交额为0或缺失# - 节假日：中国A股有法定节假日休市# - 数据源问题：baostock接口偶发返回空值

### 2.2 缺失值处理

In [ ]:
# 缺失值处理策略：向前填充（ffill）# 选择依据：股价数据具有时间连续性，停牌期间价格不变，ffill是最合理的处理方式for code, df in stock_data.items():    before = df.isna().sum().sum()    df = df.fillna(method='ffill')    # 如果首行缺失，用后向填充    df = df.fillna(method='bfill')    after = df.isna().sum().sum()    stock_data[code] = df    if before > 0:        print(f"  {code}: 填充 {before} 个缺失值 → 剩余 {after}")print("\n缺失值处理完成！")

### 2.3 日期格式统一

In [ ]:
# 统一日期为 datetime64 格式，并设为索引for code in stock_data:    stock_data[code]["日期"] = pd.to_datetime(stock_data[code]["日期"])    stock_data[code] = stock_data[code].set_index("日期").sort_index()hs300["日期"] = pd.to_datetime(hs300["日期"])hs300 = hs300.set_index("日期").sort_index()zz500["日期"] = pd.to_datetime(zz500["日期"])zz500 = zz500.set_index("日期").sort_index()cpi["日期"] = pd.to_datetime(cpi["日期"])m2["日期"] = pd.to_datetime(m2["日期"])print("日期格式统一完成！")print(f"示例索引类型: {stock_data['600036'].index.dtype}")

### 2.4 数据类型检查

In [ ]:
# 确认价格、成交量列为数值型type_report = []for code, df in stock_data.items():    for col in ["开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额"]:        dtype = str(df[col].dtype)        if "object" in dtype:            df[col] = pd.to_numeric(df[col], errors="coerce")            type_report.append(f"  {code} {col}: {dtype} → numeric")            if type_report:    print("类型转换记录：")    for r in type_report:        print(r)else:    print("所有数值列类型正确，无需转换")

### 2.5 重复值处理

In [ ]:
# 检测并删除重复行total_removed = 0for code, df in stock_data.items():    n_before = len(df)    df = df[~df.index.duplicated(keep='first')]    n_removed = n_before - len(df)    total_removed += n_removed    stock_data[code] = df    if n_removed > 0:        print(f"  {code}: 删除 {n_removed} 条重复记录")print(f"\n共删除 {total_removed} 条重复记录")

### 2.6 离群值标注

In [ ]:
# 计算日收益率，标注单日涨跌幅超过±20%的记录for code, df in stock_data.items():    df["日收益率"] = np.log(df["收盘价"] / df["收盘价"].shift(1))    df["is_extreme"] = df["日收益率"].abs() > 0.20    n_extreme = df["is_extreme"].sum()    stock_data[code] = df    if n_extreme > 0:        print(f"  {code}: {n_extreme} 个极端值")        # 显示极端值日期        extreme_dates = df[df["is_extreme"]].index.tolist()        for d in extreme_dates[:3]:            print(f"    {d.strftime('%Y-%m-%d')}: 收盘价={df.loc[d, '收盘价']:.2f}, 收益率={df.loc[d, '日收益率']*100:.2f}%")# 可能成因：# 1. 复权因子调整：后复权数据在除权除息日会出现跳涨# 2. ST股涨跌停：极端行情下的连续涨跌停# 3. 停牌复牌：长期停牌后复牌首日可能出现大幅波动print("\n离群值标注完成！（不删除，仅标注）")

## 3. 宽表与长表转换

In [ ]:
# 3.1 宽表：日期为索引，每列一只股票的收盘价wide_df = pd.DataFrame()for code, df in stock_data.items():    wide_df[code] = df["收盘价"]wide_df.columns = [f"{s['code']}_{s['name']}" for s in STOCKS]print("宽表（收盘价）：")print(wide_df.head())print(f"shape: {wide_df.shape}")

In [ ]:
# 3.2 长表：用 pd.melt 转换long_df = wide_df.reset_index().melt(    id_vars=["日期"],     var_name="code_name",     value_name="close")long_df[["code", "name"]] = long_df["code_name"].str.split("_", n=1, expand=True)long_df = long_df[["日期", "code", "name", "close"]].dropna()long_df.columns = ["date", "code", "name", "close"]print("长表：")print(long_df.head(10))print(f"shape: {long_df.shape}")

In [ ]:
### 宽表 vs 长表的适用场景**宽表适合**：- 同时比较多只股票（如绘制多股票走势图）- 计算股票间的相关系数矩阵- 做面板数据的截面分析**长表适合**：- 按股票分组做分组统计（groupby）- 存入数据库（关系型数据库更偏好长格式）- 做混合效应回归、面板回归等

## 4. 多表合并

In [ ]:
# 4.1 个股日度数据与指数日度数据按日期 left join# 先准备个股长表（含行业信息）all_stock_list = []for code, df in stock_data.items():    stock_info = next(s for s in STOCKS if s["code"] == code)    temp = df[["开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额", "日收益率", "is_extreme"]].copy()    temp["code"] = code    temp["name"] = stock_info["name"]    temp["industry"] = stock_info["industry"]    all_stock_list.append(temp)stock_long = pd.concat(all_stock_list, axis=0)stock_long = stock_long.reset_index()print(f"合并前个股数据: {stock_long.shape}")# 合并沪深300hs300_merge = hs300[["收盘价"]].rename(columns={"收盘价": "hs300_close"})stock_merged = stock_long.merge(hs300_merge, left_on="日期", right_index=True, how="left")print(f"合并沪深300后: {stock_merged.shape}")# 合并中证500zz500_merge = zz500[["收盘价"]].rename(columns={"收盘价": "zz500_close"})stock_merged = stock_merged.merge(zz500_merge, left_on="日期", right_index=True, how="left")print(f"合并中证500后: {stock_merged.shape}")

In [ ]:
# 4.2 月度宏观数据与日度数据合并# 处理频率不一致：将CPI和M2映射到对应月份的每个交易日cpi["year_month"] = cpi["日期"].dt.to_period("M")m2["year_month"] = m2["日期"].dt.to_period("M")stock_merged["year_month"] = pd.to_datetime(stock_merged["日期"]).dt.to_period("M")# 合并CPIcpi_map = cpi[["year_month", "CPI同比"]].drop_duplicates(subset=["year_month"])stock_merged = stock_merged.merge(cpi_map, on="year_month", how="left")# 合并M2m2_map = m2[["year_month", "M2同比"]].drop_duplicates(subset=["year_month"])stock_merged = stock_merged.merge(m2_map, on="year_month", how="left")# 记录合并前后行数print(f"最终合并数据: {stock_merged.shape}")print(f"行数变化说明：left join 保持日度数据的行数不变，新增宏观数据列")

## 5. 存储清洗后数据

### 5.1 方式A：CSV存储

In [ ]:
clean_dir = os.path.join(BASE_DIR, "data", "clean")combined_dir = os.path.join(BASE_DIR, "data", "combined")os.makedirs(clean_dir, exist_ok=True)os.makedirs(combined_dir, exist_ok=True)# 保存个股清洗后数据（长表）stock_merged.to_csv(os.path.join(clean_dir, "stock_clean.csv"), index=False, encoding="utf-8-sig")print(f"stock_clean.csv 已保存: {stock_merged.shape}")# 保存宽表wide_df.to_csv(os.path.join(clean_dir, "wide_close.csv"), encoding="utf-8-sig")print(f"wide_close.csv 已保存: {wide_df.shape}")# 保存合并后综合数据stock_merged.to_csv(os.path.join(combined_dir, "combined_data.csv"), index=False, encoding="utf-8-sig")print(f"combined_data.csv 已保存: {stock_merged.shape}")

### 5.2 方式C：SQLite存储（进阶）

In [ ]:
# 创建SQLite数据库，设计3张表db_path = os.path.join(combined_dir, "fin_data.db")if os.path.exists(db_path):    os.remove(db_path)conn = sqlite3.connect(db_path)# 表1: stock_price — 股票日度行情stock_price = stock_merged[["日期", "code", "name", "industry",                              "开盘价", "收盘价", "最高价", "最低价", "成交量", "成交额",                              "日收益率", "is_extreme"]].copy()stock_price.columns = ["date", "code", "name", "industry",                         "open", "close", "high", "low", "volume", "amount",                        "return", "is_extreme"]stock_price.to_sql("stock_price", conn, if_exists="replace", index=False)# 表2: macro_data — 宏观经济数据（长格式）macro_long = pd.concat([    cpi[["日期", "CPI同比"]].rename(columns={"日期": "date", "CPI同比": "value"}).assign(indicator="cpi"),    m2[["日期", "M2同比"]].rename(columns={"日期": "date", "M2同比": "value"}).assign(indicator="m2")], ignore_index=True)macro_long["date"] = pd.to_datetime(macro_long["date"])macro_long.to_sql("macro_data", conn, if_exists="replace", index=False)# 表3: stock_info — 股票基本信息stock_info_df = pd.DataFrame(STOCKS)stock_info_df.to_sql("stock_info", conn, if_exists="replace", index=False)conn.commit()print("SQLite数据库创建完成！")print(f"数据库文件: {db_path}")

In [ ]:
# 演示跨表JOINquery = """SELECT p.date, p.code, s.name, s.industry, p.close,        m.value AS cpiFROM stock_price pLEFT JOIN macro_data m       ON substr(p.date, 1, 7) = substr(m.date, 1, 7)      AND m.indicator = 'cpi'LEFT JOIN stock_info s       ON p.code = s.codeLIMIT 10"""result = pd.read_sql_query(query, conn)print("跨表JOIN示例：")result

In [ ]:
# 自定义SQL查询1：查询各行业股票的年均收盘价query1 = """SELECT s.industry, p.code, s.name,        AVG(p.close) AS avg_close,       MIN(p.close) AS min_close,       MAX(p.close) AS max_closeFROM stock_price pJOIN stock_info s ON p.code = s.codeWHERE p.close > 0GROUP BY s.industry, p.codeORDER BY s.industry, avg_close DESC"""result1 = pd.read_sql_query(query1, conn)print("查询1：各行业股票年均收盘价")result1

In [ ]:
# 自定义SQL查询2：查询CPI最高月份的各行业平均收益率query2 = """SELECT s.industry,        AVG(p.return) AS avg_return,       COUNT(*) AS trading_daysFROM stock_price pJOIN stock_info s ON p.code = s.codeJOIN macro_data m ON substr(p.date, 1, 7) = substr(m.date, 1, 7) AND m.indicator = 'cpi'WHERE m.value = (SELECT MAX(value) FROM macro_data WHERE indicator = 'cpi')GROUP BY s.industryORDER BY avg_return DESC"""result2 = pd.read_sql_query(query2, conn)print("查询2：CPI最高月份各行业平均收益率")result2

In [ ]:
conn.close()print("数据库操作完成！")

## 6. CSV vs SQLite 对比说明| 特性 | CSV | SQLite ||------|-----|--------|| 通用性 | 极强，任何工具可读 | 需SQLite驱动，但Python内置支持 || 查询能力 | 需pandas操作 | 支持SQL，跨表JOIN效率高 || 类型约束 | 无 | 有Schema约束 || 并发安全 | 差 | 支持事务 || 文件大小 | 较大 | 较小（压缩存储） || 适用场景 | 小数据、简单分析 | 多表关联、复杂查询 |**选择SQLite的理由**：本项目有多类数据（行情、指数、宏观、财务）需要关联查询，SQLite的跨表JOIN能力远胜于多次读取CSV后pandas merge，且数据库文件体积更小、查询更灵活。